In [1]:
from pathlib import Path
import pandas as pd

In [2]:
Path("../transactions.parquet").stat().st_size / 1024 / 1024
Path("../transactions_25M.parquet").stat().st_size / 1024 / 1024
Path("../transactions_50M.parquet").stat().st_size / 1024 / 1024

1194.6014966964722

In [10]:

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.read.parquet("../transactions.parquet")
print(df.rdd.getNumPartitions())

8


In [7]:
summary = pd.read_csv(Path("../results/sweep_20261012-162458/sweep_summary.csv"))
summary["framework"] = summary["framework"].str.replace(" (local[*])", "", regex=False)

# fastest framework per benchmark at each size
fastest = summary.loc[summary.groupby(["benchmark", "size_m"])["median_s"].idxmin(),
                    ["benchmark", "size_m", "framework", "median_s", "rows_per_sec"]]
# print(fastest.sort_values(["benchmark", "size_m"]))
print(summary.pivot_table(index=["benchmark", "size_m"],
                            columns="framework",
                            values="median_s").round(2))

framework         Pandas  Polars  Spark
benchmark size_m                       
Aggregate 10        0.95    0.08   0.36
          25        4.33    0.18   0.49
          50       29.31    0.60   1.16
Ingest    10        1.67    0.15   0.71
          25        4.99    7.10   0.98
          50       24.64   30.57   3.55
Transform 10        0.78    0.05   0.50
          25        3.72    0.13   0.69
          50       27.41    0.25   3.84


In [8]:
summary = pd.read_csv(Path("../results/sweep_20261012-162458/sweep_summary.csv"))
summary["framework"] = summary["framework"].str.replace(" (local[*])", "", regex=False)

min_times = summary.groupby(["benchmark", "size_m"])["median_s"].transform("min")
summary["vs_best"] = (summary["median_s"] / min_times).round(2)

best_fw = summary.loc[summary.groupby(["benchmark", "size_m"])["median_s"].idxmin(),
                    ["benchmark", "size_m", "framework"]].rename(columns={"framework": "best"})
summary = summary.merge(best_fw, on=["benchmark", "size_m"])

print(summary[["benchmark", "size_m", "framework", "median_s", "vs_best", "best"]]
    .sort_values(["benchmark", "size_m", "framework"]))

    benchmark  size_m framework  median_s  vs_best    best
7   Aggregate      10    Pandas    0.9498    11.98  Polars
6   Aggregate      10    Polars    0.0793     1.00  Polars
8   Aggregate      10     Spark    0.3608     4.55  Polars
16  Aggregate      25    Pandas    4.3274    24.13  Polars
15  Aggregate      25    Polars    0.1793     1.00  Polars
17  Aggregate      25     Spark    0.4881     2.72  Polars
25  Aggregate      50    Pandas   29.3146    48.70  Polars
24  Aggregate      50    Polars    0.6020     1.00  Polars
26  Aggregate      50     Spark    1.1587     1.92  Polars
1      Ingest      10    Pandas    1.6723    11.43  Polars
0      Ingest      10    Polars    0.1463     1.00  Polars
2      Ingest      10     Spark    0.7131     4.87  Polars
10     Ingest      25    Pandas    4.9913     5.11   Spark
9      Ingest      25    Polars    7.1002     7.27   Spark
11     Ingest      25     Spark    0.9760     1.00   Spark
19     Ingest      50    Pandas   24.6372     6.93   Spa